In [5]:
from typing import Annotated, TypedDict, List
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from mem0 import MemoryClient
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from dotenv import load_dotenv
import os
DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
MEM0_API_KEY = os.getenv('MEM0_API_KEY')
load_dotenv()


D:\PyCharm\PROJECT\RAGProject\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
D:\PyCharm\PROJECT\RAGProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


False

In [7]:
llm = ChatOpenAI(
    temperature=0.5,
    model='deepseek-chat',
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com")
mem0 = MemoryClient(
    api_key=MEM0_API_KEY
)

In [8]:
class State(TypedDict):
    messages: Annotated[List[HumanMessage | AIMessage], add_messages]
    mem0_user_id: str

graph = StateGraph(State)

def chatbot(state: State):
    messages = state["messages"]
    user_id = state["mem0_user_id"]

    try:
        # 检索相关记忆
        # memories = mem0.search(messages[-1].content,user_id=user_id)
        memories = mem0.search(messages[-1].content,
                               filters={"user_id": user_id},
                               version="v2")

        # 处理字典响应格式
        memory_list = memories['results']

        context = "来自先前对话的相关信息：\n"
        for memory in memory_list:
            context += f"- {memory['memory']}\n"

        system_message = SystemMessage(content=f"""你是一个乐于助人的客户支持助理。使用提供的上下文来个性化你的回复，并记住用户的偏好和过去的互动。
{context}""")

        full_messages = [system_message] + messages
        response = llm.invoke(full_messages)

        # 将交互存储在Mem0中
        try:
            interaction = [
                {
                    "role": "user",
                    "content": messages[-1].content
                },
                {
                    "role": "assistant",
                    "content": response.content
                }
            ]
            result = mem0.add(interaction, user_id=user_id)
            print(f"记忆已保存：已添加 {len(result.get('results', []))} 条记忆")
        except Exception as e:
            print(f"保存记忆时出错：{e}")

        return {"messages": [response]}

    except Exception as e:
        print(f"聊天机器人出错：{e}")
        # 无记忆上下文的备选回复
        response = llm.invoke(messages)
        return {"messages": [response]}

In [9]:
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", "chatbot")

compiled_graph = graph.compile()

In [10]:
def run_conversation(user_input: str, mem0_user_id: str):
    config = {"configurable": {"thread_id": mem0_user_id}}
    state = {"messages": [HumanMessage(content=user_input)], "mem0_user_id": mem0_user_id}

    for event in compiled_graph.stream(state, config):
        for value in event.values():
            if value.get("messages"):
                print("客户支持：", value["messages"][-1].content)
                return

In [11]:
print("欢迎使用客户支持！我今天能为您提供什么帮助？")
mem0_user_id = "alice"  # 您可以根据您的用户管理系统生成或检索此ID
while True:
    user_input = input("您：")
    if user_input.lower() in ['quit', 'exit', 'bye']:
        print("客户支持：感谢您联系我们。祝您有美好的一天！")
        break
    run_conversation(user_input, mem0_user_id)

欢迎使用客户支持！我今天能为您提供什么帮助？
记忆已保存：已添加 1 条记忆
客户支持： 黄帅，你好！很高兴认识你！有什么我可以帮助你的吗？
记忆已保存：已添加 1 条记忆
客户支持： 根据我们的对话记录，您之前介绍过自己是**黄帅**。有什么我可以帮您的吗？ 😊
记忆已保存：已添加 1 条记忆
客户支持： 很高兴认识你，黄帅！打篮球是一项很棒的运动，既能锻炼身体又能培养团队精神。你平时喜欢打什么位置呢？
记忆已保存：已添加 1 条记忆
客户支持： 根据之前的对话，您提到自己是**黄帅**（Huang Shuai）。不过，关于您的喜好，目前还没有具体信息。如果您愿意分享，我很乐意记住您的兴趣，以便未来为您提供更个性化的帮助！ 😊
记忆已保存：已添加 1 条记忆
客户支持： 根据我们之前的对话，你提到过你喜欢打篮球！这项运动确实充满活力，既能锻炼身体，又能培养团队合作精神。最近有去打篮球吗？或者有没有尝试过其他相关的运动呢？ 😊


KeyboardInterrupt: Interrupted by user